# Fine-tuning Phi-4-mini-instruct for Goal Decomposition
**Model:** `microsoft/Phi-4-mini-instruct`  
**Method:** QLoRA (4-bit) + SFT via LLaMA Factory  
**Task:** Decompose user goals into structured, actionable tasks (JSON output)  
**Hardware:** Kaggle 2× T4 GPU

---
## Notebook Structure
1. [Environment Setup](#1-environment-setup)
2. [Authentication](#2-authentication)
3. [Dataset Preparation](#3-dataset-preparation)
4. [LLaMA Factory Configuration](#4-llamafactory-configuration)
5. [Training](#5-training)
6. [Save Checkpoints to HuggingFace](#6-save-checkpoints-to-huggingface)
7. [Model Loading for Inference](#7-model-loading-for-inference)
8. [Evaluation Utilities](#8-evaluation-utilities)
9. [Run Evaluation](#9-run-evaluation)


## 1. Environment Setup

In [ ]:
# Install LLaMA Factory and dependencies
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e ".[torch,metrics]" -q
!pip install -q wandb transformers==4.51.3 json-repair tqdm bert-score sentence-transformers


## 2. Authentication

In [ ]:
import os
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Load secrets
secrets   = UserSecretsClient()
hf_token  = secrets.get_secret('huggingface')
wandb_key = secrets.get_secret('wandb')

# Login
login(token=hf_token)
wandb.login(key=wandb_key)

# Set environment variables
os.environ["HF_TOKEN"]              = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True"

# Enable both GPUs
os.environ.pop("CUDA_VISIBLE_DEVICES", None)

# Verify HF login
!hf auth whoami


## 3. Dataset Preparation

In [ ]:
import json
import os
import random
from os.path import join

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_DIR  = "/kaggle/input/datasets/eslamadly666/complete-goal-dataset"
DATA_DIR   = "/kaggle/working/decomposition_llm"
TRAIN_PATH = join(DATA_DIR, "datasets", "llamafactory-finetune-data", "train.json")
VAL_PATH   = join(DATA_DIR, "datasets", "llamafactory-finetune-data", "val.json")

os.makedirs(join(DATA_DIR, "datasets", "llamafactory-finetune-data"), exist_ok=True)


In [ ]:
# ── System prompt ─────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a smart goal decomposition assistant. "
    "Given a user's goal and personal context, decompose it into specific, actionable tasks. "
    "Always respond with a valid JSON object only, no extra text or explanation.\n\n"
    "Follow these rules strictly:\n"
    "- If goal_type is 'habit': ALL tasks must be repeatable and must include a repeat_frequency\n"
    "- If goal_type is 'one_time': ALL tasks must be non-repeatable\n"
    "- Rest or gap days are only applicable to habit goals\n"
    "- If user constraints are provided, always respect them when generating tasks"
)

# ── Conversion function ────────────────────────────────────────────────────
def convert_sample(sample: dict) -> dict:
    """Convert raw dataset sample to LLaMA Factory Alpaca format."""
    instruction = (
        f"Goal: {sample['goal']}\n"
        f"Category: {sample['goal_category']}\n"
        f"Type: {sample['goal_type']}\n"
        f"Time Horizon: {sample['time_horizon']} days"
    )
    return {
        "system":      SYSTEM_PROMPT,
        "instruction": instruction,
        "input":        "",
        "output":       json.dumps({"tasks": sample["tasks"]}, ensure_ascii=False),
        "history":      []
    }


In [ ]:
# ── Load, convert, and split dataset ──────────────────────────────────────
with open(join(INPUT_DIR, "complete_dataset.jsonl")) as f:
    raw_samples = [json.loads(line) for line in f if line.strip()]

converted = [convert_sample(s) for s in raw_samples]
random.Random(101).shuffle(converted)

split      = int(len(converted) * 0.95)
train_data = converted[:split]
eval_data  = converted[split:]

with open(TRAIN_PATH, "w") as f:
    json.dump(train_data, f, ensure_ascii=False, default=str)

with open(VAL_PATH, "w", encoding="utf8") as f:
    json.dump(eval_data, f, ensure_ascii=False, default=str)

print(f"Train samples : {len(train_data)}")
print(f"Val samples   : {len(eval_data)}")
print(f"\nSample preview:\n{json.dumps(converted[0], indent=2, ensure_ascii=False)[:500]}...")


In [ ]:
# ── Register datasets with LLaMA Factory ──────────────────────────────────
DATASET_INFO_PATH = "/kaggle/working/LLaMA-Factory/data/dataset_info.json"

with open(DATASET_INFO_PATH) as f:
    dataset_info = json.load(f)

COLUMN_MAP = {
    "prompt":   "instruction",
    "query":    "input",
    "response": "output",
    "system":   "system",
    "history":  "history"
}

dataset_info["goals_finetune_train"] = {"file_name": TRAIN_PATH, "columns": COLUMN_MAP}
dataset_info["goals_finetune_val"]   = {"file_name": VAL_PATH,   "columns": COLUMN_MAP}

with open(DATASET_INFO_PATH, "w") as f:
    json.dump(dataset_info, f, indent=2)

print("dataset_info.json updated successfully")


### 3.1 Token Length Analysis
Run this cell to find the right `cutoff_len` for your data.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-4-mini-instruct",
    trust_remote_code=True,
    cache_dir="/kaggle/working/hf_cache"
)

with open(TRAIN_PATH) as f:
    data = json.load(f)

lengths = sorted([
    tokenizer(s["system"] + s["instruction"] + s["output"], return_tensors="pt")["input_ids"].shape[1]
    for s in data
])

total = len(lengths)
print(f"Min  : {lengths[0]}")
print(f"Mean : {int(sum(lengths)/total)}")
print(f"p75  : {lengths[int(total*0.75)]}")
print(f"p90  : {lengths[int(total*0.90)]}")
print(f"p95  : {lengths[int(total*0.95)]}")
print(f"p99  : {lengths[int(total*0.99)]}")
print(f"Max  : {lengths[-1]}")
print("\nRecommended cutoff_len: p95 value")


## 4. LLaMA Factory Configuration
Edit the YAML below to change training settings.  
Key parameters:
- `cutoff_len` — set to p95 token length from analysis above
- `num_train_epochs` — increase for more training (use with `load_best_model_at_end`)
- `export_hub_model_id` — your HuggingFace repo for checkpoints


In [ ]:
%%writefile /kaggle/working/LLaMA-Factory/examples/train_lora/goal_decompose.yaml
### model
model_name_or_path: microsoft/Phi-4-mini-instruct
trust_remote_code: true
quantization_bit: 4
quantization_method: bitsandbytes

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all

### dataset
dataset: goals_finetune_train
eval_dataset: goals_finetune_val
template: phi4_mini
cutoff_len: 1200
overwrite_cache: true
preprocessing_num_workers: 4
dataloader_num_workers: 4

### output
output_dir: /kaggle/working/decomposition_llm/llm_checkpoints_v3
logging_steps: 100
save_steps: 100
plot_loss: true
overwrite_output_dir: false
save_only_model: false
save_total_limit: 20

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 10.0
lr_scheduler_type: cosine
warmup_steps: 30
fp16: true
ddp_timeout: 180000000
upcast_layernorm: true

### early stopping
load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false

### eval
per_device_eval_batch_size: 2
eval_strategy: steps
eval_steps: 100

### logging
report_to: wandb
run_name: goal_decomposition-finetune-v3

### hub
push_to_hub: true
export_hub_model_id: "Eslam666/goal_decomposition_llm_v3"
hub_private_repo: true
hub_strategy: checkpoint


## 5. Training

In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /kaggle/working/LLaMA-Factory/examples/train_lora/goal_decompose.yaml


## 6. Save Checkpoints to HuggingFace
Run this **immediately after training** before closing the Kaggle session.  
This pushes every local checkpoint to HuggingFace so you can load them later.


In [ ]:
from huggingface_hub import HfApi, create_repo

CHECKPOINT_DIR = "/kaggle/working/decomposition_llm/llm_checkpoints_v3"
HF_REPO_ID     = "Eslam666/goal_decomposition_llm_v3"

api = HfApi(token=hf_token)

# Create repo if it doesn't exist
create_repo(repo_id=HF_REPO_ID, private=True, token=hf_token, exist_ok=True)

# Push all checkpoint folders
checkpoints = sorted([
    d for d in os.listdir(CHECKPOINT_DIR)
    if d.startswith("checkpoint-")
])
print(f"Found {len(checkpoints)} checkpoints: {checkpoints}")

for ckpt in checkpoints:
    local_path = os.path.join(CHECKPOINT_DIR, ckpt)
    print(f"Pushing {ckpt}...")
    api.upload_folder(
        folder_path  = local_path,
        repo_id      = HF_REPO_ID,
        path_in_repo = ckpt,
        token        = hf_token
    )
    print(f"  ✅ {ckpt} pushed!")

print("\nAll checkpoints saved to HuggingFace!")


## 7. Model Loading for Inference
Load the fine-tuned model from HuggingFace.  
Change `CHECKPOINT` to load a specific checkpoint (e.g. `"checkpoint-500"`).  
Set `CHECKPOINT = None` to load the final model.


In [ ]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH  = "microsoft/Phi-4-mini-instruct"
ADAPTER_REPO = "Eslam666/goal_decomposition_llm_v3"
CHECKPOINT   = None   # e.g. "checkpoint-500", or None for final model

# ── Load tokenizer ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# ── Load base model ────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ── Load LoRA adapter ──────────────────────────────────────────────────────
load_kwargs = {"token": hf_token}
if CHECKPOINT:
    load_kwargs["subfolder"] = CHECKPOINT

model = PeftModel.from_pretrained(model, ADAPTER_REPO, **load_kwargs)
model.eval()

checkpoint_label = CHECKPOINT or "final"
print(f"✅ Loaded: {ADAPTER_REPO} [{checkpoint_label}]")
print(f"   Device map: {model.hf_device_map}")


### 7.1 Verify GPU Usage

In [ ]:
# Verify model is on GPU
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU 0 memory  : {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
if torch.cuda.device_count() > 1:
    print(f"GPU 1 memory  : {torch.cuda.memory_allocated(1)/1024**3:.2f} GB")
print(f"Model device  : {next(model.parameters()).device}")


## 8. Evaluation Utilities

In [ ]:
import json
import torch
from tqdm import tqdm
from statistics import mean
from bert_score import score as bert_score_fn
from sentence_transformers import SentenceTransformer, util

# ── Sentence embedder for semantic similarity ──────────────────────────────
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
# ── Generation function ───────────────────────────────────────────────────
def generate(goal, category, goal_type, time_horizon, constraints=None):
    """Generate decomposed tasks for a goal using the fine-tuned model."""
    instruction = (
        f"Goal: {goal}\n"
        f"Category: {category}\n"
        f"Type: {goal_type}\n"
        f"Time Horizon: {time_horizon} days"
    )
    if constraints:
        instruction += f"\nConstraints:\n- Physical: {constraints}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": instruction}
    ]

    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = {k: v.to(model.device) for k, v in tokenizer(text, return_tensors="pt").items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 1024,
            temperature    = 0.7,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
            use_cache      = True
        )

    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────
def tasks_to_text(tasks):
    """Flatten task list to a single string for BERTScore comparison."""
    return " | ".join(
        f"{t.get('task', '')} {t.get('description', '')}".strip()
        for t in tasks
    )

def semantic_task_similarity(gt_tasks, pred_tasks, use_description=True):
    """
    Bidirectional semantic similarity between ground truth and predicted tasks.
    
    Returns dict with precision, recall, f1:
    - Precision: are generated tasks relevant to ground truth?
    - Recall:    did model cover all ground truth tasks?
    - F1:        harmonic mean of precision and recall
    """
    def get_texts(tasks):
        if use_description:
            return [f"{t.get('task', '')} {t.get('description', '')}".strip() for t in tasks]
        return [t.get("task", "") for t in tasks]

    gt_texts, pred_texts = get_texts(gt_tasks), get_texts(pred_tasks)

    if not gt_texts or not pred_texts:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    gt_emb   = embedder.encode(gt_texts,   convert_to_tensor=True)
    pred_emb = embedder.encode(pred_texts, convert_to_tensor=True)

    precision = mean([util.cos_sim(p, gt_emb).max().item()   for p in pred_emb])
    recall    = mean([util.cos_sim(g, pred_emb).max().item() for g in gt_emb])
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {"precision": precision, "recall": recall, "f1": f1}

def check_rule_compliance(goal_type, pred_tasks):
    """Check if generated tasks follow habit/one_time rules."""
    if not pred_tasks:
        return False
    if goal_type == "habit":
        return all(
            t.get("is_repeatable") == True and t.get("repeat_frequency", 0) > 0
            for t in pred_tasks
        )
    return all(t.get("is_repeatable") == False for t in pred_tasks)


In [ ]:
# ── Main evaluation pipeline ──────────────────────────────────────────────
def run_evaluation(samples, generate_fn, use_description=True,
                   bert_model="distilbert-base-uncased", bert_batch_size=16, label="Model"):
    """
    Full evaluation pipeline. Accepts two input formats:

    Format 1 (raw dataset):
        {"Goal": ..., "Category": ..., "Type": ..., "Time Horizon": ..., "output": "...json..."}

    Format 2 (pre-evaluated):
        {"input": {...}, "ground_truth": {...}, "prediction": {...}}

    Args:
        samples      : list of dicts in either format
        generate_fn  : callable(goal, category, goal_type, time_horizon) -> str
                       Pass None to use existing predictions (Format 2 only)
        label        : name shown in the summary report

    Returns:
        results  : list of per-sample dicts with all metrics attached
        summary  : dict of aggregate metrics
    """
    results, bert_preds, bert_refs = [], [], []
    invalid_json = 0

    for sample in tqdm(samples, desc=f"Evaluating [{label}]"):

        # Normalize input format
        if "Goal" in sample:
            goal_info = {
                "goal":         sample["Goal"],
                "category":     sample["Category"],
                "type":         sample["Type"],
                "time_horizon": sample["Time Horizon"]
            }
            try:
                ground_truth = json.loads(sample["output"])
            except json.JSONDecodeError:
                invalid_json += 1
                continue
        else:
            goal_info    = sample["input"]
            ground_truth = sample["ground_truth"]

        # Generate or use existing prediction
        if generate_fn is not None:
            raw = generate_fn(goal_info["goal"], goal_info["category"],
                              goal_info["type"],  goal_info["time_horizon"])
            # Strip markdown wrapping (base model sometimes adds it)
            raw = raw.strip().replace("```json", "").replace("```", "").strip()
            try:
                prediction = json.loads(raw)
            except json.JSONDecodeError:
                invalid_json += 1
                prediction = {"tasks": []}
        else:
            prediction = sample.get("prediction", {"tasks": []})

        gt_tasks   = ground_truth.get("tasks", [])
        pred_tasks = prediction.get("tasks", []) if isinstance(prediction, dict) else prediction

        # Normalize prediction to dict
        if isinstance(prediction, list):
            prediction = {"tasks": prediction}
            pred_tasks = prediction["tasks"]

        results.append({
            "input":           goal_info,
            "ground_truth":    ground_truth,
            "prediction":      prediction,
            "similarity":      semantic_task_similarity(gt_tasks, pred_tasks, use_description),
            "rule_compliance": check_rule_compliance(goal_info["type"], pred_tasks),
            "task_count_ok":   abs(len(pred_tasks) - len(gt_tasks)) <= 1,
        })

        if gt_tasks and pred_tasks:
            bert_preds.append(tasks_to_text(pred_tasks))
            bert_refs.append(tasks_to_text(gt_tasks))

    # BERTScore (batch computation)
    if bert_preds:
        print(f"\nComputing BERTScore for {len(bert_preds)} samples...")
        P, R, F1 = bert_score_fn(
            bert_preds, bert_refs,
            lang="en", model_type=bert_model,
            batch_size=bert_batch_size, verbose=True
        )
        bert_f1_list = F1.tolist()
        bert_p_list  = P.tolist()
        bert_r_list  = R.tolist()
    else:
        bert_f1_list = bert_p_list = bert_r_list = []

    # Attach BERTScore to each result
    bert_idx = 0
    for r in results:
        if r["ground_truth"].get("tasks") and r["prediction"].get("tasks"):
            r["bert_score"] = {
                "precision": bert_p_list[bert_idx],
                "recall":    bert_r_list[bert_idx],
                "f1":        bert_f1_list[bert_idx]
            }
            bert_idx += 1
        else:
            r["bert_score"] = {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    # Build summary
    valid   = [r for r in results if r["prediction"].get("tasks")]
    n_total = len(results)
    n_habit = sum(1 for r in valid if r["input"]["type"] == "habit")
    n_one   = sum(1 for r in valid if r["input"]["type"] == "one_time")

    summary = {
        "label":         label,
        "total":         n_total,
        "valid_json":    n_total - invalid_json,
        "invalid_json":  invalid_json,
        "task_count_ok": sum(r["task_count_ok"] for r in valid),
        "habit_ok":      sum(1 for r in valid if r["input"]["type"] == "habit"    and r["rule_compliance"]),
        "habit_total":   n_habit,
        "onetime_ok":    sum(1 for r in valid if r["input"]["type"] == "one_time" and r["rule_compliance"]),
        "onetime_total": n_one,
        "similarity": {
            "precision": mean([r["similarity"]["precision"] for r in valid]) if valid else 0,
            "recall":    mean([r["similarity"]["recall"]    for r in valid]) if valid else 0,
            "f1":        mean([r["similarity"]["f1"]        for r in valid]) if valid else 0,
            "min":       min( [r["similarity"]["f1"]        for r in valid]) if valid else 0,
            "max":       max( [r["similarity"]["f1"]        for r in valid]) if valid else 0,
        },
        "bert_score": {
            "precision": mean(bert_p_list)  if bert_p_list else 0,
            "recall":    mean(bert_r_list)  if bert_r_list else 0,
            "f1":        mean(bert_f1_list) if bert_f1_list else 0,
        }
    }

    _print_summary(summary, results, bert_f1_list)
    return results, summary


In [ ]:
# ── Summary printer ──────────────────────────────────────────────────────
def _print_summary(summary, results, bert_f1_list):
    n     = summary["valid_json"]
    valid = [r for r in results if r["prediction"].get("tasks")]

    print(f"\n{'='*55}")
    print(f"  EVALUATION REPORT — {summary['label']}")
    print(f"{'='*55}")

    print(f"\n── Dataset ──────────────────────────────────────")
    print(f"  Total samples    : {summary['total']}")
    print(f"  Valid JSON        : {summary['valid_json']}")
    print(f"  Invalid JSON      : {summary['invalid_json']}")

    print(f"\n── BERTScore ────────────────────────────────────")
    print(f"  Precision        : {summary['bert_score']['precision']:.4f}")
    print(f"  Recall           : {summary['bert_score']['recall']:.4f}")
    print(f"  F1               : {summary['bert_score']['f1']:.4f}")

    print(f"\n── Semantic Similarity ──────────────────────────")
    print(f"  Precision        : {summary['similarity']['precision']:.4f}")
    print(f"  Recall           : {summary['similarity']['recall']:.4f}")
    print(f"  F1 (mean)        : {summary['similarity']['f1']:.4f}")
    print(f"  F1 (min)         : {summary['similarity']['min']:.4f}")
    print(f"  F1 (max)         : {summary['similarity']['max']:.4f}")

    print(f"\n── Structural Metrics ───────────────────────────")
    print(f"  Task count OK    : {summary['task_count_ok']}/{n} ({100*summary['task_count_ok']//n if n else 0}%)")

    print(f"\n── Rule Compliance ──────────────────────────────")
    if summary["habit_total"] > 0:
        print(f"  Habit rules OK   : {summary['habit_ok']}/{summary['habit_total']} ({100*summary['habit_ok']//summary['habit_total']}%)")
    if summary["onetime_total"] > 0:
        print(f"  One-time rules OK: {summary['onetime_ok']}/{summary['onetime_total']} ({100*summary['onetime_ok']//summary['onetime_total']}%)")

    if bert_f1_list:
        print(f"\n── Worst 5 Samples (BERTScore F1) ───────────────")
        for idx in sorted(range(len(bert_f1_list)), key=lambda i: bert_f1_list[i])[:5]:
            r = valid[idx]
            print(f"  [{idx:3d}] BERT F1: {bert_f1_list[idx]:.4f} | Sim F1: {r['similarity']['f1']:.4f}")
            print(f"        Goal : {r['input']['goal'][:70]}")
            print(f"        Pred : {[t['task'] for t in r['prediction']['tasks']]}")
            print(f"        GT   : {[t['task'] for t in r['ground_truth']['tasks']]}\n")

        print(f"── Best 5 Samples (BERTScore F1) ────────────────")
        for idx in sorted(range(len(bert_f1_list)), key=lambda i: bert_f1_list[i], reverse=True)[:5]:
            r = valid[idx]
            print(f"  [{idx:3d}] BERT F1: {bert_f1_list[idx]:.4f} | Sim F1: {r['similarity']['f1']:.4f}")
            print(f"        Goal : {r['input']['goal'][:70]}\n")


def save_results(results, path):
    """Save evaluation results to JSONL file."""
    with open(path, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
    print(f"Saved {len(results)} results → {path}")


def print_comparison_table(summaries: dict):
    """Print multiple evaluation summaries side by side."""
    labels = list(summaries.keys())

    print(f"\n{'='*75}")
    print(f"  COMPARISON TABLE")
    print(f"{'='*75}")
    header = f"  {'Metric':<25}"
    for label in labels:
        header += f" {label[:13]:>13}"
    print(header)
    print(f"  {'-'*70}")

    def row(name, fn):
        line = f"  {name:<25}"
        for label in labels:
            line += f" {fn(summaries[label]):>13}"
        print(line)

    h = lambda s: s["habit_total"]   or 1
    o = lambda s: s["onetime_total"] or 1

    row("Valid JSON %",      lambda s: f"{100*s['valid_json']//s['total']}%")
    row("BERTScore F1",      lambda s: f"{s['bert_score']['f1']:.4f}")
    row("BERTScore P",       lambda s: f"{s['bert_score']['precision']:.4f}")
    row("BERTScore R",       lambda s: f"{s['bert_score']['recall']:.4f}")
    row("Similarity F1",     lambda s: f"{s['similarity']['f1']:.4f}")
    row("Similarity P",      lambda s: f"{s['similarity']['precision']:.4f}")
    row("Similarity R",      lambda s: f"{s['similarity']['recall']:.4f}")
    row("Task Count OK %",   lambda s: f"{100*s['task_count_ok']//s['total']}%")
    row("Habit Rules %",     lambda s: f"{100*s['habit_ok']//h(s)}%")
    row("One-time Rules %",  lambda s: f"{100*s['onetime_ok']//o(s)}%")


## 9. Run Evaluation

### 9.1 Load Test Data


In [ ]:
INPUT_DIR = "/kaggle/input/datasets/eslamadly666/complete-goal-dataset"

with open(join(INPUT_DIR, "data_test_dataset.jsonl")) as f:
    data_test_dataset = [json.loads(line) for line in f if line.strip()]

with open(join(INPUT_DIR, "gemini_test_dataset.jsonl")) as f:
    gemini_test_dataset = [json.loads(line) for line in f if line.strip()]

print(f"Test samples   : {len(data_test_dataset)}")
print(f"Gemini samples : {len(gemini_test_dataset)}")
print(f"\nSample keys: {list(data_test_dataset[0].keys())}")


### 9.2 Quick Inference Test

In [ ]:
# Quick sanity check before full evaluation
result = generate(
    goal         = "Run 5km every morning",
    category     = "fitness",
    goal_type    = "habit",
    time_horizon = 30
)
try:
    parsed = json.loads(result)
    print(f"✅ Valid JSON | {len(parsed['tasks'])} tasks generated")
    for t in parsed["tasks"]:
        print(f"  - {t['task']} | repeatable={t['is_repeatable']} | freq={t.get('repeat_frequency', 'N/A')}")
except json.JSONDecodeError:
    print("❌ Invalid JSON output:")
    print(result[:500])


### 9.3 Fine-tuned Model Evaluation

In [ ]:
# Evaluate fine-tuned model on test dataset
ft_results, ft_summary = run_evaluation(
    data_test_dataset,
    generate_fn = generate,
    label       = "Fine-tuned (v3)"
)

save_results(ft_results, "finetuned_results.jsonl")


### 9.4 Base Model Evaluation (Baseline Comparison)

In [ ]:
# ── Base model system prompt — includes full schema since base model needs guidance ──
BASE_SYSTEM_PROMPT = (
    "You are a smart goal decomposition assistant. "
    "Given a user's goal and personal context, decompose it into specific, actionable tasks. "
    "Always respond with a valid JSON object only, no extra text, no markdown, no code blocks.\n\n"
    "Follow these rules strictly:\n"
    "- If goal_type is 'habit': ALL tasks must have is_repeatable=true and repeat_frequency > 0\n"
    "- If goal_type is 'one_time': ALL tasks must have is_repeatable=false and repeat_frequency=0\n"
    "- gap_flag should be true only for rest days in habit goals\n"
    "- estimated_duration_minutes must have min and max values in minutes\n\n"
    "Required JSON schema:\n"
    '{"tasks": [{"index": 1, "task": "...", "description": "...", '
    '"is_repeatable": true, "repeat_frequency": 1, "gap_flag": false, '
    '"estimated_duration_minutes": {"min": 20, "max": 30}}]}'
)

def generate_base(goal, category, goal_type, time_horizon, constraints=None):
    """Generate using base model (no LoRA adapter) for baseline comparison."""
    instruction = (
        f"Goal: {goal}\n"
        f"Category: {category}\n"
        f"Type: {goal_type}\n"
        f"Time Horizon: {time_horizon} days"
    )
    if constraints:
        instruction += f"\nConstraints: {constraints}"

    messages = [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user",   "content": instruction}
    ]

    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = {k: v.to(base_model.device) for k, v in tokenizer(text, return_tensors="pt").items()}

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens = 1024,
            temperature    = 0.7,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
            use_cache      = True
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
# Load base model (no adapter)
import gc
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()
print("✅ Base model loaded")


In [ ]:
# Evaluate base model
base_results, base_summary = run_evaluation(
    data_test_dataset,
    generate_fn = generate_base,
    label       = "Base Model"
)

save_results(base_results, "base_model_results.jsonl")


### 9.5 Comparison: Base vs Fine-tuned

In [ ]:
print_comparison_table({
    "Base Model":      base_summary,
    "Fine-tuned (v3)": ft_summary,
})


### 9.6 Additional Experiments

Run these for richer analysis in your graduation project documentation.


In [ ]:
from collections import defaultdict

def evaluate_by_category(results, label=""):
    """Break down BERTScore and similarity by goal category."""
    categories = defaultdict(list)
    for r in results:
        categories[r["input"]["category"]].append(r)

    title = f"PERFORMANCE BY CATEGORY" + (f" — {label}" if label else "")
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(f"  {'Category':<18} {'N':>5} {'BERT F1':>9} {'Sim F1':>9} {'Rules%':>8}")
    print(f"  {'-'*55}")

    for cat, rs in sorted(categories.items()):
        bert_f1 = mean([r["bert_score"]["f1"]    for r in rs])
        sim_f1  = mean([r["similarity"]["f1"]     for r in rs])
        rules   = sum(1 for r in rs if r["rule_compliance"])
        print(f"  {cat:<18} {len(rs):>5} {bert_f1:>9.4f} {sim_f1:>9.4f} {100*rules//len(rs):>7}%")


def evaluate_by_type(results, label=""):
    """Break down metrics by goal type (habit vs one_time)."""
    title = f"PERFORMANCE BY GOAL TYPE" + (f" — {label}" if label else "")
    print(f"\n{'='*55}\n  {title}\n{'='*55}")

    for goal_type in ["habit", "one_time"]:
        rs = [r for r in results if r["input"]["type"] == goal_type]
        if not rs:
            continue
        bert_f1 = mean([r["bert_score"]["f1"] for r in rs])
        sim_f1  = mean([r["similarity"]["f1"]  for r in rs])
        rules   = sum(1 for r in rs if r["rule_compliance"])
        print(f"\n  {goal_type.upper()} ({len(rs)} samples)")
        print(f"    BERTScore F1    : {bert_f1:.4f}")
        print(f"    Similarity F1   : {sim_f1:.4f}")
        print(f"    Rule Compliance : {rules}/{len(rs)} ({100*rules//len(rs)}%)")


def evaluate_by_time_horizon(results, label=""):
    """Break down metrics by time horizon length."""
    buckets = {
        "Short  (1–7 days)":   [r for r in results if r["input"]["time_horizon"] <= 7],
        "Medium (8–30 days)":  [r for r in results if 8  <= r["input"]["time_horizon"] <= 30],
        "Long   (31–180 days)": [r for r in results if r["input"]["time_horizon"]  > 30],
    }
    title = f"PERFORMANCE BY TIME HORIZON" + (f" — {label}" if label else "")
    print(f"\n{'='*55}\n  {title}\n{'='*55}")

    for bucket_label, rs in buckets.items():
        if not rs:
            continue
        bert_f1 = mean([r["bert_score"]["f1"] for r in rs])
        sim_f1  = mean([r["similarity"]["f1"]  for r in rs])
        print(f"\n  {bucket_label} — {len(rs)} samples")
        print(f"    BERTScore F1  : {bert_f1:.4f}")
        print(f"    Similarity F1 : {sim_f1:.4f}")


In [ ]:
# Run all breakdown analyses on fine-tuned results
evaluate_by_category(ft_results,      label="Fine-tuned")
evaluate_by_type(ft_results,          label="Fine-tuned")
evaluate_by_time_horizon(ft_results,  label="Fine-tuned")
